# Views: story plan & pipeline

Branch-tab support views for the app's `/story/pipeline` endpoint. Split out of
`story_plan_chapters_load_v2` (Section 2). Each is cheap to compute and always reflects the
latest `gold_person_scores` / `silver_person_story_status`, so a view keeps them live.

## Why views live in their own notebooks
`CREATE OR REPLACE VIEW` drops and recreates the view, which makes the **executing identity its
owner**. A view owned by someone else that reads it (e.g. `gold_person_evidence_summary`) then
fails with `INSUFFICIENT_PERMISSIONS`. So view DDL is kept out of the data-refresh notebooks:
views are *definitions*, not data, and only need (re)creating when a definition changes.

**Run this notebook only when a view definition changes**, not as part of a routine refresh.
Run it as the account that owns the views (or follow it with
`ALTER VIEW <name> OWNER TO <owner>` for each view it recreates).

Depends on: `silver_story_plan_chapters` (loaded by `story_plan_chapters_load_v2`), `silver_person_story_status`, `gold_person_life`, `gold_person_branch`, `gold_person_scores`.


In [0]:
%sql
-- One row per (chapter, person) — expands person_gedcom_ids and resolves names.
-- CTE explodes first, then joins — LATERAL VIEW EXPLODE must precede any join
-- on serverless, so exploding inside a CTE and joining the result normally
-- avoids that constraint entirely.
CREATE OR REPLACE VIEW genealogy.gold_story_plan_people AS
WITH exploded AS (
  SELECT chapter_number, part_name, title, status, flags, doc_version,
         explode(person_gedcom_ids) AS person_gedcom_id
  FROM genealogy.silver_story_plan_chapters
)
SELECT e.chapter_number, e.part_name, e.title, e.status, e.flags, e.doc_version,
       e.person_gedcom_id, pl.display_name, pl.birth_year, pl.death_year, pb.branch
FROM exploded e
LEFT JOIN genealogy.gold_person_life pl ON pl.person_gedcom_id = e.person_gedcom_id
LEFT JOIN genealogy.gold_person_branch pb ON pb.person_gedcom_id = e.person_gedcom_id;


In [0]:
%sql
-- Expect one row per (chapter, person) pair — 60 chapters, several with 2+ people.
-- Row count should exceed 60; spot-check a known multi-person chapter.
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT chapter_number) AS distinct_chapters
FROM genealogy.gold_story_plan_people;


In [0]:
%sql
-- Everyone with any story-workflow progress (mid-pipeline or written),
-- joined to life/branch/score context. Powers the "In Progress" section
-- of a branch's story pipeline.
CREATE OR REPLACE VIEW genealogy.gold_story_pipeline AS
SELECT s.person_gedcom_id, s.story_stage, s.story_written, s.story_title,
       s.story_url, s.published_at, s.timeline_doc_id, s.context_doc_id,
       s.draft_doc_id, s.story_doc_id, s.notes, s.updated_at,
       pl.display_name, pl.birth_year, pl.death_year, pb.branch,
       ps.story_potential_score, ps.overall_score
FROM genealogy.silver_person_story_status s
LEFT JOIN genealogy.gold_person_life pl ON pl.person_gedcom_id = s.person_gedcom_id
LEFT JOIN genealogy.gold_person_branch pb ON pb.person_gedcom_id = s.person_gedcom_id
LEFT JOIN genealogy.gold_person_scores ps ON ps.person_gedcom_id = s.person_gedcom_id
WHERE s.story_stage != 'not_started' OR s.story_written = true;


In [0]:
%sql
-- Expect 20 rows (row count of silver_person_story_status at time of writing
-- that meet the WHERE clause) — adjust expectation as more people enter the pipeline.
SELECT branch, COUNT(*) AS people_in_pipeline
FROM genealogy.gold_story_pipeline
GROUP BY branch
ORDER BY branch;


In [0]:
%sql
-- Story-ready candidates NOT already covered by the plan or the pipeline,
-- top 3 per branch by story potential. Powers "Other Candidates" — people
-- with strong scores who aren't part of the Sixty Stories plan.
CREATE OR REPLACE VIEW genealogy.gold_story_candidates_by_branch AS
WITH planned AS (
  SELECT explode(person_gedcom_ids) AS person_gedcom_id
  FROM genealogy.silver_story_plan_chapters
),
in_pipeline AS (
  SELECT person_gedcom_id FROM genealogy.silver_person_story_status
  WHERE story_stage != 'not_started' OR story_written = true
),
ranked AS (
  SELECT ps.person_gedcom_id, ps.display_name, ps.branch, ps.birth_year,
         ps.death_year, ps.story_potential_score,
         ROW_NUMBER() OVER (PARTITION BY ps.branch ORDER BY ps.story_potential_score DESC) AS rn
  FROM genealogy.gold_person_scores ps
  WHERE ps.story_ready = true
    AND ps.story_written = false
    AND ps.branch IS NOT NULL
    AND ps.person_gedcom_id NOT IN (SELECT person_gedcom_id FROM planned)
    AND ps.person_gedcom_id NOT IN (SELECT person_gedcom_id FROM in_pipeline)
)
SELECT person_gedcom_id, display_name, branch, birth_year, death_year, story_potential_score
FROM ranked
WHERE rn <= 3;


In [0]:
%sql
-- Expect at most 3 rows per branch — anomaly check for the window function, expect 0
SELECT branch, COUNT(*) AS candidate_count
FROM genealogy.gold_story_candidates_by_branch
GROUP BY branch
HAVING COUNT(*) > 3;


In [0]:
%sql
-- Spot-check: nobody in this view should also appear in the plan or pipeline — expect 0
SELECT COUNT(*) AS overlap_count
FROM genealogy.gold_story_candidates_by_branch c
WHERE c.person_gedcom_id IN (
  SELECT explode(person_gedcom_ids) FROM genealogy.silver_story_plan_chapters
) OR c.person_gedcom_id IN (
  SELECT person_gedcom_id FROM genealogy.silver_person_story_status
  WHERE story_stage != 'not_started' OR story_written = true
);
